# Piso 3ºB · Voces con Chatterbox España (todo automático)
Hace solo **todo lo que antes tenías que hacer a mano**:
1. Crea una voz de referencia para cada personaje (con las voces de España de Microsoft Edge, ajustando tono y ritmo para que sean distintas).
2. Se las manda a la demo de **Chatterbox España** (la que te gustó, en Hugging Face) para que hable con ese timbre y acento de España.
3. Te muestra cada audio con un **número grande** para que me digas cuáles te convencen.

*Entorno de ejecución → Ejecutar todo.* No hace falta GPU: el modelo corre en los servidores de Hugging Face.

In [ ]:
!pip install -q -U gradio_client edge-tts

In [ ]:
# Opcional pero recomendable: un token GRATIS de Hugging Face da más tiempo de GPU al día.
# huggingface.co/settings/tokens -> New token (tipo Read). Pégalo aquí entre comillas. Si lo dejas vacío, funciona con el límite anónimo.
HF_TOKEN = ""
SPACE = "ResembleAI/Chatterbox-Multilingual-TTS-es-es"

In [ ]:
import subprocess, os, re, json
from gradio_client import Client, handle_file
from IPython.display import Audio, display, HTML
os.makedirs("refs", exist_ok=True); os.makedirs("salida", exist_ok=True)

# --- 1) Voces de España disponibles en Edge (gratis, sin clave) ---
lista = subprocess.run(["edge-tts", "--list-voices"], capture_output=True, text=True).stdout
es = [l.split()[0:2] for l in lista.splitlines() if l.startswith("es-ES-")]
H = [n for n, g in es if g == "Male"]; M = [n for n, g in es if g == "Female"]
print("Voces es-ES:", H, M)
# personaje -> (voz base, tono, ritmo, idioma de la referencia)
PERS = {
 "yo":     (H[0],            "-10Hz", "-5%",  "es"),
 "amigo":  (H[-1],           "+0Hz",  "+0%",  "es"),
 "amiga":  (M[0],            "+0Hz",  "+0%",  "es"),
 "jefe":   (H[0],            "-30Hz", "-12%", "es"),
 "abuela": (M[-1],           "+25Hz", "-18%", "es"),
 "madre":  (M[0],            "-12Hz", "+0%",  "es"),
 "emily":  ("en-US-AriaNeural", "+0Hz", "+0%", "en"),   # americana: referencia en inglés
}
TXT = {"es": "Hola, buenas. Pues mira, yo creo que esto es lo que hay, y no hay mucho más que hablar. Vosotros veréis lo que hacéis, la verdad.",
       "en": "Hi, how are you doing today? I think this is going to be a really great day, and I am very happy to be here."}
for k, (v, p, r, lg) in PERS.items():
    subprocess.run(["edge-tts", "--voice", v, f"--pitch={p}", f"--rate={r}", "--text", TXT[lg], "--write-media", f"refs/{k}.mp3"], check=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", f"refs/{k}.mp3", "-ar", "24000", "-ac", "1", f"refs/{k}.wav"], check=True)
print("Referencias creadas:", list(PERS))

In [ ]:
# --- 2) Conectar con la demo de Chatterbox España y ver sus parámetros ---
client = Client(SPACE, token=HF_TOKEN or None) if HF_TOKEN else Client(SPACE)
api = client.view_api(return_format="dict")
ep = list(api["named_endpoints"].items())[0]
print("Endpoint:", ep[0]); 
for p in ep[1]["parameters"]: print(" -", p.get("parameter_name"), "|", p.get("python_type", {}).get("type"), "| defecto:", p.get("parameter_default"))

In [ ]:
# --- 3) Función para generar: rellena los parámetros por su nombre ---
def hablar(texto, ref, exag=0.5):
    kw = {}
    for p in ep[1]["parameters"]:
        n = p["parameter_name"]; ln = n.lower(); d = p.get("parameter_default")
        if ln in ("text", "text_input", "texto", "prompt_text") or ("text" in ln and "ref" not in ln and "prompt" not in ln): kw[n] = texto[:300]
        elif "audio" in ln or "ref" in ln or "prompt" in ln or "voice" in ln: kw[n] = handle_file(ref)
        elif "exag" in ln: kw[n] = exag
        elif "seed" in ln: kw[n] = 0
        elif "randomize" in ln: kw[n] = False
        elif d is not None: kw[n] = d
    res = client.predict(api_name=ep[0], **kw)
    if isinstance(res, (list, tuple)): res = next(x for x in res if isinstance(x, str))
    return res
FRASE = {
 "yo":     "Son las ocho y ocho, y ya llego tarde otra vez.",
 "amigo":  "Tranquilo, tío, que sobra tiempo. ¿Quién ha dejado la sartén en el suelo?",
 "amiga":  "No pasa nada, es puntualidad. Pero la nevera ha hecho un ruido muy raro.",
 "jefe":   "Esto es un piso, no un hotel. Quiero el alquiler antes del lunes, ¿vale?",
 "abuela": "Ay, hijo, qué ruido hacéis. En mis tiempos esto no pasaba.",
 "madre":  "Cariño, ¿has comido algo? Que te veo muy delgado, de verdad.",
 "emily":  "Hola, chicos. Yo no entiendo por qué la nevera hace ese ruido tan raro.",
}

In [ ]:
# --- 4) Generar con 3 niveles de emoción («exaggeration») y mostrar con número ---
import shutil
n = 0
for k in PERS:
    for exag, nombre in [(0.3, "tranquilo"), (0.6, "normal"), (0.95, "muy expresivo")]:
        n += 1
        try:
            f = hablar(FRASE[k], f"refs/{k}.wav", exag)
            dst = f"salida/{n:02d}_{k}_{nombre}.wav"; shutil.copy(f, dst)
            display(HTML(f"<div style='font:700 28px sans-serif;margin-top:14px'>Nº {n:02d} · {k} · {nombre}</div>")); display(Audio(dst))
        except Exception as e:
            print(f"Nº {n:02d} {k} {nombre}: ERROR ->", str(e)[:300]); 

In [ ]:
!zip -qr salida.zip salida
from google.colab import files; files.download("salida.zip")

## Qué me tienes que decir
- ¿Suena **a España** en todos?
- Los **números** de los que te convencen, y qué te parece el de **emily** (americana hablando español).
- Si sale algún **ERROR** en rojo o en una línea «ERROR ->», cópiamelo tal cual.